In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Categorical

from cs590_env.combat_env import get_card_mask, mask_logits
from cs590_src.combat_agent_model import CombatPPOAgent
from cs590_src.ppo_config import PPOConfig

In [2]:
def compute_log_prob_and_entropy(
    agent: CombatPPOAgent,
    obs_batch: dict,
    card_sels: torch.Tensor,
    executions: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Forward the agent and compute log-probs / entropy for stored actions.

    Works on any batch size B (the flattened T*N during PPO, or N during rollout).

    Returns:
        log_probs: (B,)
        entropy:   (B,)
        values:    (B,)
    """
    sel_logits, exec_logits, values = agent(obs_batch)
    card_mask = get_card_mask(obs_batch)
    sel_logits = mask_logits(sel_logits, card_mask)

    sel_dist  = Categorical(logits=sel_logits)
    exec_dist = Categorical(logits=exec_logits)

    log_probs = (sel_dist.log_prob(card_sels).sum(dim=-1)
                 + exec_dist.log_prob(executions))
    entropy   = (sel_dist.entropy().sum(dim=-1)
                 + exec_dist.entropy())
    return log_probs, entropy, values.squeeze(-1)


def ppo_update(
    agent: CombatPPOAgent,
    optimizer: torch.optim.Optimizer,
    obs_batch: dict,
    card_sels: torch.Tensor,
    executions: torch.Tensor,
    old_log_probs: torch.Tensor,
    advantages: torch.Tensor,
    returns: torch.Tensor,
    cfg: PPOConfig,
) -> dict:
    """Run PPO clipped-surrogate epochs over flattened (T*N) rollout data.

    Returns:
        dict of mean losses for logging.
    """
    B = len(advantages)
    mb_size = B // cfg.num_minibatches
    total_pg, total_vf, total_ent = 0.0, 0.0, 0.0
    num_updates = 0

    for _ in range(cfg.ppo_epochs):
        indices = torch.randperm(B, device=advantages.device)

        for start in range(0, B, mb_size):
            end = start + mb_size
            if end > B:
                break
            mb = indices[start:end]

            mb_obs          = {k: v[mb] for k, v in obs_batch.items()}
            mb_card_sels    = card_sels[mb]
            mb_executions   = executions[mb]
            mb_old_lp       = old_log_probs[mb]
            mb_adv          = advantages[mb]
            mb_ret          = returns[mb]

            curr_lp, entropy, curr_values = compute_log_prob_and_entropy(
                agent, mb_obs, mb_card_sels, mb_executions)

            ratio = torch.exp(curr_lp - mb_old_lp)
            surr1 = ratio * mb_adv
            surr2 = torch.clamp(ratio, 1 - cfg.clip_eps, 1 + cfg.clip_eps) * mb_adv
            pg_loss    = -torch.min(surr1, surr2).mean()
            value_loss = nn.functional.mse_loss(curr_values, mb_ret)
            ent_bonus  = entropy.mean()

            loss = pg_loss + cfg.c_value * value_loss - cfg.c_entropy * ent_bonus

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(agent.parameters(), cfg.max_grad_norm)
            optimizer.step()

            total_pg  += pg_loss.item()
            total_vf  += value_loss.item()
            total_ent += ent_bonus.item()
            num_updates += 1

    return {
        'pg_loss':    total_pg  / max(num_updates, 1),
        'value_loss': total_vf  / max(num_updates, 1),
        'entropy':    total_ent / max(num_updates, 1),
    }